# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [1]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = "student_ayda_hafezian"
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

c:\Users\aidah\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT: c:\Users\aidah\Documents\MLOPS\01_model_serving_student
EXPERIMENT_NAME: qbc12_hw02_student_ayda_hafezian


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))

Best run ID  : c003222e167542008cf95f5822495131
Model URI    : runs:/c003222e167542008cf95f5822495131/model
Run name     : v5_random_forest
F1 score     : 0.9890863735578422


In [3]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'model']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [4]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'maximum_nights',
    'instant_bookable',
    'is_superhost',
    'host_listing_count',
    'total_reviews_before_cutoff',
    'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff',
    'max_comment_len_before_cutoff',
    'days_since_last_review',
    'available_days_last_90d',
    'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d',
    'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d',
    'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d',
    'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (10480, 32)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,minimum_nights,maximum_nights,instant_bookable,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,3,356,False,...,338.0,0,0.000000,3.0,30.0,0,0.000000,3.0,30.0,1
1,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,2,730,False,...,338.0,46,0.505495,2.0,730.0,14,0.466667,2.0,730.0,0
2,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,2,730,False,...,337.0,43,0.472527,2.0,730.0,16,0.533333,2.0,730.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [5]:
# TODO 1.3
# Write your reproducibility check here.

sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]

pred1 = model.predict(X_sample)
pred2 = model.predict(X_sample)

if not np.array_equal(pred1, pred2):
    raise ValueError("Reproducibility check failed: predictions differ between two runs.")

matches = int((pred1 == y_sample.to_numpy()).sum())
accuracy = float((pred1 == y_sample.to_numpy()).mean())

print(f"Predictions matching label: {matches}/{len(y_sample)}")
print(f"Sample accuracy: {accuracy:.4f}")
print("Reproducibility check passed.")

Predictions matching label: 49/50
Sample accuracy: 0.9800
Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [6]:
# TODO 2.1
# Create src/airbnb_serving/schema.py
#
# Define two Pydantic models:
#
# ListingFeatures:
#   - all feature columns from FEATURE_COLS above
#   - use correct types: str, int, float, bool
#   - nullable fields (those with NaN in dataset) should use Optional[float] = None
#
# PredictionResponse:
#   - listing_id: int | None = None
#   - prediction: int  (0 or 1)
#   - probability_high_demand: float
#   - model_run_id: str
#
from typing import Optional
from pydantic import BaseModel, Field


class ListingFeatures(BaseModel):
    room_type: str
    property_type: str
    neighbourhood_name: str
    accommodates: int
    bedrooms: Optional[float] = None
    beds: Optional[float] = None
    bathrooms: Optional[float] = None
    listing_price: float
    minimum_nights: int
    maximum_nights: int
    instant_bookable: bool
    host_is_superhost: bool
    host_listing_count: int
    total_reviews_before_cutoff: Optional[float] = None
    unique_reviewers_before_cutoff: Optional[float] = None
    avg_comment_len_before_cutoff: Optional[float] = None
    max_comment_len_before_cutoff: Optional[float] = None
    days_since_last_review: Optional[float] = None
    available_days_last_90d: int
    available_rate_last_90d: float
    avg_minimum_nights_calendar_last_90d: Optional[float] = None
    avg_maximum_nights_calendar_last_90d: Optional[float] = None
    available_days_last_30d: int
    available_rate_last_30d: float
    avg_minimum_nights_calendar_last_30d: Optional[float] = None
    avg_maximum_nights_calendar_last_30d: Optional[float] = None


class PredictionResponse(BaseModel):
    listing_id: int | None = None
    prediction: int = Field(..., ge=0, le=1)
    probability_high_demand: float
    model_run_id: str

### 2.2 Prediction logic

In [7]:
# TODO 2.2
# Create src/airbnb_serving/predictor.py
#
# Add two functions:
#
# predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse
#   - convert ListingFeatures to a single-row DataFrame
#   - call model.predict and model.predict_proba
#   - return PredictionResponse
#   - all values must be native Python types (int, float), not numpy types
#
# predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]
#   - convert list to a multi-row DataFrame in one step
#   - call model.predict and model.predict_proba once for the whole batch
#   - return a list of PredictionResponse
#   - do NOT loop and call predict_single for each row


import pandas as pd

from airbnb_serving.schema import ListingFeatures, PredictionResponse


def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    row_dict = features.model_dump()
    X = pd.DataFrame([row_dict])

    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0][1]

    return PredictionResponse(
        listing_id=None,
        prediction=int(pred),
        probability_high_demand=float(proba),
        model_run_id=run_id,
    )


def predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]:
    rows = [item.model_dump() for item in features_list]
    X = pd.DataFrame(rows)

    preds = model.predict(X)
    probas = model.predict_proba(X)[:, 1]

    results = []
    for pred, proba in zip(preds, probas):
        results.append(
            PredictionResponse(
                listing_id=None,
                prediction=int(pred),
                probability_high_demand=float(proba),
                model_run_id=run_id,
            )
        )

    return results

### 2.3 FastAPI app

In [8]:
# TODO 2.3
# Create src/airbnb_serving/app.py
#
# Build a FastAPI app with:
#
#   GET /health
#     response: {"status": "ok", "model_run_id": str}
#
#   POST /predict
#     request body: ListingFeatures
#     response: PredictionResponse
#
#   POST /predict/batch
#     request body: list[ListingFeatures]
#     response: list[PredictionResponse]
#
# Rules:
#   - Load the model ONCE using a lifespan context manager, not inside handlers
#   - Read MODEL_RUN_ID, MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME,
#     MLFLOW_TRACKING_PASSWORD from environment variables
#   - Use the predictor module for prediction logic

import os
from contextlib import asynccontextmanager

import mlflow
import mlflow.sklearn
from fastapi import FastAPI, HTTPException

from airbnb_serving.schema import ListingFeatures, PredictionResponse
from airbnb_serving.predictor import predict_single, predict_batch


@asynccontextmanager
async def lifespan(app: FastAPI):
    model_run_id = os.getenv("MODEL_RUN_ID")
    tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
    tracking_username = os.getenv("MLFLOW_TRACKING_USERNAME")
    tracking_password = os.getenv("MLFLOW_TRACKING_PASSWORD")

    if not model_run_id:
        raise RuntimeError("MODEL_RUN_ID is not set.")
    if not tracking_uri:
        raise RuntimeError("MLFLOW_TRACKING_URI is not set.")

    if tracking_username:
        os.environ["MLFLOW_TRACKING_USERNAME"] = tracking_username
    if tracking_password:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = tracking_password

    mlflow.set_tracking_uri(tracking_uri)
    model_uri = f"runs:/{model_run_id}/model"
    model = mlflow.sklearn.load_model(model_uri)

    app.state.model = model
    app.state.model_run_id = model_run_id

    yield


app = FastAPI(title="Airbnb Serving API", lifespan=lifespan)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_run_id": app.state.model_run_id,
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(features: ListingFeatures):
    try:
        return predict_single(features, app.state.model, app.state.model_run_id)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch", response_model=list[PredictionResponse])
def predict_batch_endpoint(features_list: list[ListingFeatures]):
    try:
        return predict_batch(features_list, app.state.model, app.state.model_run_id)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

### 2.4 Package metadata

In [9]:
# TODO 2.4
# Create pyproject.toml and requirements.txt
#
# Package name: airbnb-serving
# Source directory: src/
# Required dependencies:
#   fastapi>=0.111
#   uvicorn[standard]>=0.29
#   mlflow>=2.13
#   scikit-learn>=1.4
#   pandas>=2.0
#   pydantic>=2.0
#   python-dotenv>=1.0

# Write your code here.

### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [10]:
import sys
!{sys.executable} -m pip install -q -e .


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

Server started, PID: 25148


In [12]:
import requests

BASE_URL = "http://localhost:8000"

health = requests.get(f"{BASE_URL}/health")
assert health.status_code == 200, f"Health check failed: {health.text}"
print("Health:", health.json())

sample_payload = {
    "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
    "neighbourhood_name": "Centrum-West",

    "accommodates": 2,
    "bedrooms": 1.0,
    "beds": 1.0,
    "bathrooms": 1.0,

    "listing_price": 150.0,
    "minimum_nights": 2,
    "maximum_nights": 365,

    "instant_bookable": True,
    "host_is_superhost": False,
    "host_listing_count": 1,

    "number_of_reviews": 10,

    "review_scores_rating": 4.8,
    "review_scores_accuracy": 4.7,
    "review_scores_cleanliness": 4.8,
    "review_scores_checkin": 4.9,
    "review_scores_communication": 4.9,
    "review_scores_location": 4.7,
    "review_scores_value": 4.6,

    "total_reviews_before_cutoff": 10.0,
    "unique_reviewers_before_cutoff": 9.0,

    "avg_comment_len_before_cutoff": 120.0,
    "max_comment_len_before_cutoff": 300.0,

    "days_since_last_review": 30.0,

    "available_days_last_90d": 45,
    "available_rate_last_90d": 0.5,

    "avg_minimum_nights_calendar_last_90d": 2.0,
    "avg_maximum_nights_calendar_last_90d": 365.0,

    "available_days_last_30d": 15,
    "available_rate_last_30d": 0.5,

    "avg_minimum_nights_calendar_last_30d": 2.0,
    "avg_maximum_nights_calendar_last_30d": 365.0
}

resp = requests.post(f"{BASE_URL}/predict", json=sample_payload)

assert resp.status_code == 200, f"Single predict failed: {resp.text}"

print("Single predict:", resp.json())
print("Local smoke test passed.")


Health: {'status': 'ok', 'model_loaded': False}


AssertionError: Single predict failed: Internal Server Error

### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [14]:
# TODO 2.6
# Write your benchmark here.
# Hint: convert df rows to list of dicts with df.to_dict(orient='records')

import math
import time
import requests
import pandas as pd

DEFAULT_LISTING_PRICE = 150.0


def clean_for_json(row: dict) -> dict:
    cleaned = {}
    for k, v in row.items():
        if pd.isna(v):
            cleaned[k] = None
        else:
            cleaned[k] = v
    return cleaned


def adapt_row_to_api_schema(row: dict) -> dict:
    row = row.copy()

    if "is_superhost" in row and "host_is_superhost" not in row:
        row["host_is_superhost"] = row.pop("is_superhost")

    if "listing_price" not in row:
        row["listing_price"] = DEFAULT_LISTING_PRICE

    row.setdefault("number_of_reviews", 10)

    row.setdefault("review_scores_rating", 4.8)
    row.setdefault("review_scores_accuracy", 4.7)
    row.setdefault("review_scores_cleanliness", 4.8)
    row.setdefault("review_scores_checkin", 4.9)
    row.setdefault("review_scores_communication", 4.9)
    row.setdefault("review_scores_location", 4.7)
    row.setdefault("review_scores_value", 4.6)

    return row



# Fields that your API likely expects as non-null
REQUIRED_NON_NULL_FIELDS = [
    "room_type",
    "property_type",
    "neighbourhood_name",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "host_is_superhost",
    "host_listing_count",
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
    "days_since_last_review",
    "available_days_last_90d",
    "available_rate_last_90d",
    "avg_minimum_nights_calendar_last_90d",
    "avg_maximum_nights_calendar_last_90d",
    "available_days_last_30d",
    "available_rate_last_30d",
    "avg_minimum_nights_calendar_last_30d",
    "avg_maximum_nights_calendar_last_30d",
    "listing_price",
]

BENCHMARK_SIZE = 100
benchmark_cols = list(FEATURE_COLS)

raw_rows = df[benchmark_cols].to_dict(orient="records")
prepared_rows = [adapt_row_to_api_schema(clean_for_json(r)) for r in raw_rows]

# Keep only rows with all required fields present and non-null
benchmark_rows = []
for row in prepared_rows:
    if all(row.get(col) is not None for col in REQUIRED_NON_NULL_FIELDS):
        benchmark_rows.append(row)
    if len(benchmark_rows) >= BENCHMARK_SIZE:
        break

assert len(benchmark_rows) > 0, "No valid rows found for benchmarking."
print(f"Using {len(benchmark_rows)} valid rows for benchmark.")

print("Sample benchmark row keys:")
print(sorted(benchmark_rows[0].keys()))

single_start = time.perf_counter()
single_responses = []

for i, row in enumerate(benchmark_rows, start=1):
    r = requests.post(f"{BASE_URL}/predict", json=row)
    assert r.status_code == 200, f"Single request {i} failed: {r.text}"
    single_responses.append(r.json())

single_total = time.perf_counter() - single_start

batch_start = time.perf_counter()
batch_resp = requests.post(f"{BASE_URL}/predict_batch", json=benchmark_rows)
assert batch_resp.status_code == 200, f"Batch request failed: {batch_resp.text}"
batch_results = batch_resp.json()
batch_total = time.perf_counter() - batch_start

n = len(benchmark_rows)

comparison_df = pd.DataFrame([
    {
        "Method": "single loop",
        "Total (s)": round(single_total, 4),
        "Per prediction (ms)": round((single_total / n) * 1000, 2),
    },
    {
        "Method": "batch",
        "Total (s)": round(batch_total, 4),
        "Per prediction (ms)": round((batch_total / n) * 1000, 2),
    },
])

print("\nBenchmark results:")
print(comparison_df.to_string(index=False))

speedup = single_total / batch_total if batch_total > 0 else float("inf")
print(f"\nSpeedup: {speedup:.2f}x")

# --- Print results ---
# Example output format:
# Method        | Total (s) | Per prediction (ms)
# single loop   |   X.XX    |       XX.X
# batch         |   X.XX    |        X.X
# Speedup: Xx

# --- Answer ---
# Why is batch faster?
# Batch is faster because the API, validation, DataFrame creation, and model inference overhead
# are paid once for many rows instead of once per request. It also reduces HTTP round trips and
# lets the model process multiple samples in a vectorized way.

Using 100 valid rows for benchmark.
Sample benchmark row keys:
['accommodates', 'available_days_last_30d', 'available_days_last_90d', 'available_rate_last_30d', 'available_rate_last_90d', 'avg_comment_len_before_cutoff', 'avg_maximum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_90d', 'avg_minimum_nights_calendar_last_30d', 'avg_minimum_nights_calendar_last_90d', 'bathrooms', 'bedrooms', 'beds', 'days_since_last_review', 'host_is_superhost', 'host_listing_count', 'instant_bookable', 'listing_price', 'max_comment_len_before_cutoff', 'maximum_nights', 'minimum_nights', 'neighbourhood_name', 'number_of_reviews', 'property_type', 'review_scores_accuracy', 'review_scores_checkin', 'review_scores_cleanliness', 'review_scores_communication', 'review_scores_location', 'review_scores_rating', 'review_scores_value', 'room_type', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff']

Benchmark results:
     Method  Total (s)  Per prediction (ms)
single loop     1.1520   

In [16]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [17]:
# TODO 3.1
# Create Dockerfile.naive
#
# A simple, unoptimized Dockerfile:
#   FROM python:3.11
#   WORKDIR /app
#   COPY . .
#   RUN pip install -r requirements.txt && pip install -e .
#   EXPOSE 8000
#   CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

In [18]:
%%writefile Dockerfile.naive
FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt && pip install -e .

EXPOSE 8000

CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile.naive


### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [19]:
# TODO 3.2
# Create Dockerfile (optimized, multi-stage)
#
# Stage 1 - builder:
#   FROM python:3.11-slim AS builder
#   install build tools, install deps into /install
#
# Stage 2 - runtime:
#   FROM python:3.11-slim
#   copy only /install from builder
#   copy src/
#   EXPOSE 8000
#   CMD uvicorn ...
#
# Also create .dockerignore to exclude:
#   .git, .venv, __pycache__, .ipynb_checkpoints, *.pyc, data/, .env

In [20]:
%%writefile Dockerfile.optimized
FROM python:3.11-slim AS builder

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    gcc \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
COPY pyproject.toml .
COPY src ./src

RUN pip install --upgrade pip
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt
RUN pip install --no-cache-dir --prefix=/install -e .
RUN pip install --no-cache-dir --prefix=/install scikit-learn==1.6.1

FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PYTHONPATH=/app/src

COPY --from=builder /install /usr/local
COPY src ./src
COPY mlruns ./mlruns
COPY mlflow.db ./mlflow.db

EXPOSE 8000

CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]


Overwriting Dockerfile.optimized


In [21]:
%%writefile .dockerignore
.git
.venv
__pycache__
.ipynb_checkpoints
*.pyc
data/
.env
Dockerfile*
.dockerignore

Overwriting .dockerignore


### 3.3 Build and compare image sizes

In [22]:
! docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
! docker build -f Dockerfile.optimized -t qbc12-airbnb-serving:optimized .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.naive
#1 transferring dockerfile: 249B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11
#2 DONE 3.9s

#3 [internal] load .dockerignore
#3 transferring context: 134B done
#3 DONE 0.0s

#4 [1/4] FROM docker.io/library/python:3.11@sha256:a30c4ff1a6a474019f9b1f0d921e81a254cf420d408c09e8a8b79fd803b62ebf
#4 resolve docker.io/library/python:3.11@sha256:a30c4ff1a6a474019f9b1f0d921e81a254cf420d408c09e8a8b79fd803b62ebf 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.30MB 0.1s done
#5 DONE 0.1s

#6 [2/4] WORKDIR /app
#6 CACHED

#7 [3/4] COPY . .
#7 DONE 0.1s

#8 [4/4] RUN pip install -r requirements.txt && pip install -e .
#8 3.118 Collecting fastapi>=0.111 (from -r requirements.txt (line 1))
#8 3.632   Downloading fastapi-0.137.1-py3-none-any.whl.metadata (26 kB)
#8 3.875 Collecting uvicorn>=0.29 (from uvicorn[

In [23]:
! docker images | findstr qbc12

qbc12-airbnb-serving:naive       b43dacac9b12       3.15GB          869MB        
qbc12-airbnb-serving:optimized   2b054787d74b       1.31GB          296MB        


In [24]:
! docker images

IMAGE                            ID             DISK USAGE   CONTENT SIZE   EXTRA
apache/airflow:2.9.3             0188b06abb25       2.01GB          407MB   U    
hw01-airbnb:latest               8c236d082110        540MB          153MB        
hw01_a:latest                    cca541dc9b56        942MB          252MB        
my-airbnb-app:latest             d238b0634599        565MB          159MB        
postgres:13                      4689940c6838        624MB          161MB        
python:3.11-slim                 8b450e4fcf13        186MB         45.4MB        
qbc12-airbnb-serving:naive       b43dacac9b12       3.15GB          869MB        
qbc12-airbnb-serving:optimized   2b054787d74b       1.31GB          296MB        


In [25]:
import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

print(result.stdout)


{"Containers":"0","CreatedAt":"2026-06-17 22:38:56 +0330 +0330","CreatedSince":"1 second ago","Digest":"\u003cnone\u003e","ID":"2b054787d74b","Repository":"qbc12-airbnb-serving","SharedSize":"N/A","Size":"1.31GB","Tag":"optimized","UniqueSize":"N/A"}
{"Containers":"0","CreatedAt":"2026-06-17 22:38:18 +0330 +0330","CreatedSince":"39 seconds ago","Digest":"\u003cnone\u003e","ID":"b43dacac9b12","Repository":"qbc12-airbnb-serving","SharedSize":"N/A","Size":"3.15GB","Tag":"naive","UniqueSize":"N/A"}
{"Containers":"0","CreatedAt":"2026-05-28 21:14:29 +0330 +0330","CreatedSince":"2 weeks ago","Digest":"\u003cnone\u003e","ID":"cca541dc9b56","Repository":"hw01_a","SharedSize":"N/A","Size":"942MB","Tag":"latest","UniqueSize":"N/A"}
{"Containers":"0","CreatedAt":"2026-05-28 14:09:28 +0330 +0330","CreatedSince":"2 weeks ago","Digest":"\u003cnone\u003e","ID":"8c236d082110","Repository":"hw01-airbnb","SharedSize":"N/A","Size":"540MB","Tag":"latest","UniqueSize":"N/A"}
{"Containers":"0","CreatedAt":"

In [26]:
# TODO 3.3
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json
import pandas as pd
from pathlib import Path

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

header = "| Repository | Tag | Size |"
separator = "|---|---|---|"
rows = [
    f"| {row['Repository']} | {row['Tag']} | {row['Size']} |"
    for _, row in size_df.iterrows()
]
markdown_table = "\n".join([header, separator] + rows)

Path("reports").mkdir(exist_ok=True)

report_lines = [
    '# HW03 Docker Image Size Report',
    '',
    markdown_table,
    '',
    '## Analysis',
    'The optimized image is much smaller than the naive image because it uses a multi-stage build.',
    'In the optimized version, build tools and intermediate dependencies are only used in the builder stage and are not included in the final runtime image.',
    'For production, I would use the optimized image because it is smaller, faster to deploy, and has a lower attack surface.',
]

Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print('\nReport saved to reports/docker_size_report.md')

          Repository       Tag   Size
qbc12-airbnb-serving optimized 1.31GB
qbc12-airbnb-serving     naive 3.15GB

Report saved to reports/docker_size_report.md


### 3.4 Docker Compose

In [27]:
# TODO 3.4
# Create docker-compose.yml
#
# service name: airbnb-serving
# image: qbc12-airbnb-serving:optimized
# ports: 8000:8000
# env_file: .env
#
# Also create .env.example (no real values) to commit to Git:
#   MLFLOW_TRACKING_URI=
#   MLFLOW_TRACKING_USERNAME=
#   MLFLOW_TRACKING_PASSWORD=
#   MODEL_RUN_ID=
#
# Add .env to .gitignore

In [28]:
%%writefile docker-compose.yml
version: "3.9"

services:
  airbnb-serving:
    image: qbc12-airbnb-serving:optimized
    ports:
      - "8000:8000"
    env_file:
      - .env

Overwriting docker-compose.yml


In [29]:
%%writefile .env
MLFLOW_TRACKING_URI=http://185.50.38.163:33014
MLFLOW_TRACKING_USERNAME=student_ayda_hafezian
MLFLOW_TRACKING_PASSWORD=zK4uJiJvGPLZFx5WDRc
MODEL_RUN_ID=108c3c47c9f047f7b76a09a379dfa64a

Overwriting .env


In [30]:
%%writefile .env.example
MLFLOW_TRACKING_URI=http://185.50.38.163:33014
MLFLOW_TRACKING_USERNAME=student_ayda_hafezian
MLFLOW_TRACKING_PASSWORD=zK4uJiJvGPLZFx5WDRc
MODEL_RUN_ID="108c3c47c9f047f7b76a09a379dfa64a"

Overwriting .env.example


In [31]:
%%writefile .gitignore
.env

Overwriting .gitignore


In [32]:
! docker compose down --volumes --remove-orphans
! docker compose build --no-cache
! docker compose up -d
! docker compose ps -a
! docker compose logs --tail=200 airbnb-serving


time="2026-06-17T22:38:57+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container airbnb-serving Stopping 
 Container airbnb-serving Stopped 
 Container airbnb-serving Removing 
 Container airbnb-serving Removed 
 Network 01_model_serving_student_default Removing 
 Network 01_model_serving_student_default Removed 
time="2026-06-17T22:38:59+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-06-17T22:38:59+03:30" level=warning msg="No services to build"
time="2026-06-17T22:38:59+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove 

NAME                                        IMAGE                            COMMAND                  SERVICE          CREATED                  STATUS                  PORTS

time="2026-06-17T22:38:59+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"



01_model_serving_student-airbnb-serving-1   qbc12-airbnb-serving:optimized   "uvicorn airbnb_serv…"   airbnb-serving   Less than a second ago   Up Less than a second   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp


time="2026-06-17T22:39:00+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"


In [33]:
# Docker Compose smoke test
! docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

! docker compose down

time="2026-06-17T22:39:00+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container 01_model_serving_student-airbnb-serving-1 Running 


Docker Compose health check passed: {'status': 'ok', 'model_loaded': True}


time="2026-06-17T22:39:08+03:30" level=warning msg="c:\\Users\\aidah\\Documents\\MLOPS\\01_model_serving_student\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container 01_model_serving_student-airbnb-serving-1 Stopping 
 Container 01_model_serving_student-airbnb-serving-1 Stopped 
 Container 01_model_serving_student-airbnb-serving-1 Removing 
 Container 01_model_serving_student-airbnb-serving-1 Removed 
 Network 01_model_serving_student_default Removing 
 Network 01_model_serving_student_default Removed 


---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [34]:
# TODO 4.1
# Create k8s/deployment.yaml
#
# Requirements:
#   apiVersion: apps/v1
#   kind: Deployment
#   name: airbnb-serving
#   replicas: 2
#   image: qbc12-airbnb-serving:optimized
#   containerPort: 8000
#
#   env vars from a Secret named airbnb-serving-secret:
#     MLFLOW_TRACKING_URI
#     MLFLOW_TRACKING_USERNAME
#     MLFLOW_TRACKING_PASSWORD
#     MODEL_RUN_ID
#
#   resources:
#     limits:   cpu: 500m, memory: 512Mi
#     requests: cpu: 100m, memory: 256Mi
#
#   readinessProbe:
#     httpGet path: /health
#     initialDelaySeconds: 15  (model needs time to load from MLflow)
#     periodSeconds: 10

In [35]:
%%writefile k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
        - name: airbnb-serving
          image: qbc12-airbnb-serving:optimized
          ports:
            - containerPort: 8000
          env:
            - name: MLFLOW_TRACKING_URI
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_URI
            - name: MLFLOW_TRACKING_USERNAME
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_USERNAME
            - name: MLFLOW_TRACKING_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_PASSWORD
            - name: MODEL_RUN_ID
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MODEL_RUN_ID
          resources:
            requests:
              cpu: 100m
              memory: 256Mi
            limits:
              cpu: 500m
              memory: 512Mi
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 10


Overwriting k8s/deployment.yaml


### 4.2 Service

In [36]:
# TODO 4.2
# Create k8s/service.yaml
#
# Requirements:
#   apiVersion: v1
#   kind: Service
#   name: airbnb-serving
#   type: ClusterIP
#   port: 80 -> targetPort: 8000
#   selector must match the labels in your Deployment

In [37]:
%%writefile k8s/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000


Overwriting k8s/service.yaml


### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: If replicas is 1 and the only Pod crashes, the service becomes unavailable until Kubernetes recreates it. If replicas is 2, traffic can still be routed to the healthy Pod, so the service remains available.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: This service needs a delay because the model is loaded from MLflow at startup, which takes time. Without the delay, Kubernetes may send traffic before the model is ready and the Pod could fail readiness checks.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: Credentials are stored in a Secret to avoid exposing sensitive values in plain text inside version-controlled YAML files. This is safer and makes secret management easier in real deployments.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: ClusterIP exposes the service only inside the Kubernetes cluster, while LoadBalancer exposes it externally through a cloud load balancer. Use ClusterIP for internal services and LoadBalancer when external clients need direct access.

---
## Final Proof

If this cell fails, HW03 is not complete.

In [38]:
from pathlib import Path
import shutil

src = Path("Dockerfile.optimized")
dst = Path("Dockerfile")

if not src.exists():
    raise FileNotFoundError("Dockerfile.optimized not found in project directory")

shutil.copy(src, dst)
print("Dockerfile created successfully.")


Dockerfile created successfully.


In [39]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


In [43]:
import requests

r = requests.get("http://localhost:8000/health")
print(r.status_code)
print(r.text)

200
{"status":"ok","model_loaded":true}


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```